In [169]:
#Import Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from PIL import Image

import nrrd

from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import splprep, splev

from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops
from skimage.morphology import skeletonize
from skimage.draw import line, polygon
from skimage.graph import route_through_array
from skimage.io import imsave
from skimage.util import img_as_ubyte

In [ ]:
#Load Segmentation and Set Parameters
extract_dir = r'enter_path'
#Change to GRE/ZTE_GRE/CT_GRE as needed
segmentation_path = os.path.join(extract_dir, 'GRE_Segmentation.seg.nrrd')               
output_dir = os.path.join(extract_dir, 'GRE')
os.makedirs(output_dir, exist_ok=True)

#Manually determined parameters to compute reference slice airway centerline.Adjust as needed for different subjects and different segmentation per subject.
slice_range = range(220,310)
pixel_spacing = np.array([0.48828125, 0.48828125, 0.50])  
ref_slice_idx = 250
L1 = (158, 88)
L0 = (8, 300)

# Load segmentation
seg_data_1, _ = nrrd.read(segmentation_path)
seg_data = (seg_data_1 == 1).astype(int)                                                 #Binary mask of vocal tract 

In [ ]:
#Use refernce slice to fix centerline and L0 and L1 coordinates -- Calculate only once for gre reuse geometry for zte/ct
ref_slice = seg_data[ref_slice_idx,:,:]
centerline_ref = skeletonize(ref_slice.copy(order='C'))
cost_array = np.where(centerline_ref, 1, 1e6)
start = (L0[1], L0[0])     
end   = (L1[1], L1[0])
indices, _ = route_through_array(cost_array, start, end, fully_connected=True)
path = np.array(indices)[:, [1, 0]]   
smooth_path = gaussian_filter1d(path.astype(float), sigma=2, axis=0)
tck, u = splprep([smooth_path[:, 0], smooth_path[:, 1]], s=5)

u_fine = np.linspace(0, 1, 2000)
x_f, y_f = splev(u_fine, tck)
dist = np.sqrt(np.diff(x_f)**2 + np.diff(y_f)**2)
arc = np.concatenate([[0], np.cumsum(dist)])   
arc_target = np.linspace(0, arc[-1], 40)
u_new = np.interp(arc_target, arc, u_fine)
x_new, y_new = splev(u_new, tck)
cut_points = np.vstack((x_new, y_new)).T

#Manually remove select cut points to maintain shape of vocal tract. Adjust as needed for dataset.
cut_points[38] = [162.34, 92]
cut_points[39] = [157, 87]

In [ ]:
#Calculate only once for gre reuse geometry for zte/ct
def compute_normals(path):
    tangents = np.gradient(path, axis=0)
    tangents = gaussian_filter1d(tangents, sigma=2, axis=0)
    tangents /= np.linalg.norm(tangents, axis=1, keepdims=True)
    for i in range(1, len(tangents)):
        if np.dot(tangents[i], tangents[i-1]) < 0:
            tangents[i] *= -1   
    normals = np.column_stack((-tangents[:,1], tangents[:,0]))
    normals /= np.linalg.norm(normals, axis=1, keepdims=True)
    return tangents, normals
tangents, normals = compute_normals(cut_points)

#Calculate end points of normals
slice_len = 100  
p1_list = []
p2_list = []
for pt, nrm in zip(cut_points, normals):
    p1 = pt + 0.5 * slice_len * nrm
    p2 = pt - 0.5 * slice_len * nrm
    p1_list.append(p1)
    p2_list.append(p2)
p1_array = np.array(p1_list)
p2_array = np.array(p2_list)

geometry_path = os.path.join(extract_dir, "gre_centerline_geometry.npz")
np.savez(
    geometry_path,
    cut_points=cut_points.astype(float),      
    tangents=tangents.astype(float),          
    normals=normals.astype(float),
    p1_array=p1_array.astype(float),
    p2_array=p2_array.astype(float))

In [ ]:
#Just load geometry for zte/ct to ensure consistent centerline and cut points across vocal tract models
geom_path = os.path.join(extract_dir, "gre_centerline_geometry.npz")
geom = np.load(geom_path, allow_pickle=True)
cut_points = geom["cut_points"]          
tangents   = geom["tangents"]           
normals    = geom["normals"]             
p1_array = geom["p1_array"]
p2_array = geom["p2_array"]

In [ ]:
#Calculate cross sectional area for each cut point
xy_spacing = pixel_spacing[0]
z_spacing = pixel_spacing[2]
areas_cm2 = []
for (x1, y1), (x2, y2) in zip(p1_array, p2_array):
    length = max(1, int(np.hypot(x2 - x1, y2 - y1)))
    xs = np.round(np.linspace(x1, x2, length)).astype(int)
    ys = np.round(np.linspace(y1, y2, length)).astype(int)
    xs = np.clip(xs, 0, seg_data.shape[2] - 1)
    ys = np.clip(ys, 0, seg_data.shape[1] - 1)
    slab = seg_data[slice_range][:, ys, xs]   
    count = np.sum(slab)
    area_cm2 = count * xy_spacing * z_spacing / 100.0
    areas_cm2.append(area_cm2)
areas_cm2 = np.array(areas_cm2)

#Calculate distance from L0 for each cut point
cut_points_mm = cut_points * xy_spacing
diffs = np.diff(cut_points_mm, axis=0)
segment_lengths_mm = np.sqrt(np.sum(diffs**2, axis=1))
distance_mm = np.concatenate([[0], np.cumsum(segment_lengths_mm)])
distance_cm = distance_mm / 10.0